# Gold Layer - Business KPIs and Analytical Metrics

## Objetivo

Este notebook é responsável pelo processamento da camada **Gold** do pipeline de dados, transformando os dados consolidados da Silver em indicadores de negócio (KPIs) utilizados para monitoramento operacional, análises gerenciais e tomada de decisão.

A camada Gold contém dados agregados e orientados ao negócio, representando métricas prontas para consumo por dashboards, relatórios e sistemas analíticos.

---

## Fluxo de Processamento

1. Leitura dos dados da camada Silver.
2. Aplicação das regras de negócio e cálculos de KPIs.
3. Inclusão de metadados de auditoria da Gold.
4. Gravação dos indicadores calculados na camada Gold.
5. Disponibilização dos dados para consumo analítico e carga em SQL Server.

---

# KPIs - ecommerce_clientes

## KPI 6 - Novos Clientes nos Últimos 10 Minutos

**Tipo:** Negócio

Calcula a quantidade de clientes cadastrados nos últimos 10 minutos.

**Objetivo:**

Monitorar a entrada de novos clientes em tempo real.

**Justificativa:**

Picos de cadastros podem indicar:

- Campanhas de marketing em andamento.
- Eventos promocionais.
- Comportamento anômalo causado por bots ou ataques automatizados.

**Saída Gold:**

```text
gold/ecommerce_clientes/new_customers
```

---

## KPI 7 - Distribuição por Provedor de Email

**Tipo:** Negócio

Calcula a participação percentual dos provedores de email dos clientes cadastrados.

Exemplos:

- gmail.com
- hotmail.com
- outlook.com
- yahoo.com

**Objetivo:**

Monitorar o perfil dos usuários cadastrados.

**Justificativa:**

Um aumento repentino de domínios desconhecidos ou descartáveis pode indicar:

- Cadastros automatizados.
- Fraudes.
- Criação de contas falsas.

**Saída Gold:**

```text
gold/ecommerce_clientes/email_provider_rate
```

---

## KPI 9 - Tempo Médio Entre Cadastro e Primeira Compra

**Tipo:** Negócio

Calcula o tempo médio entre:

```text
dt_cadastro
        ↓
primeira compra realizada
```

utilizando dados de clientes e pedidos.

**Objetivo:**

Mensurar a eficiência do funil de conversão.

**Justificativa:**

Aumento do tempo médio pode indicar:

- Problemas no onboarding.
- Queda na efetividade das campanhas.
- Dificuldades no processo de compra.

Reduções do indicador podem indicar melhora na conversão dos clientes.

**Saída Gold:**

```text
gold/ecommerce_clientes/avg_time_to_first_purchase
```

---

# KPIs - ecommerce_enderecos

## KPI 4 - Distribuição Geográfica de Endereços por Estado

**Tipo:** Negócio

Calcula a quantidade de novos endereços cadastrados por estado (UF).

Exemplos:

- MG
- SP
- RJ
- PR
- SC

**Objetivo:**

Monitorar a distribuição geográfica dos cadastros.

**Justificativa:**

Variações significativas podem indicar:

- Campanhas regionais.
- Expansão comercial em determinadas localidades.
- Problemas operacionais ou geração anômala de dados.

**Saída Gold:**

```text
gold/ecommerce_enderecos/address_distribution_by_state
```

---

# Estrutura da Camada Gold

A camada Gold é composta por datasets agregados e orientados ao negócio.

| Entidade | KPI | Objetivo |
|-----------|-----------|-----------|
| ecommerce_clientes | new_customers | Novos clientes nos últimos 10 minutos |
| ecommerce_clientes | email_provider_rate | Distribuição por provedor de email |
| ecommerce_clientes | avg_time_to_first_purchase | Tempo médio até a primeira compra |
| ecommerce_enderecos | address_distribution_by_state | Distribuição geográfica por estado |

---

## Observações

- O notebook é reutilizável para múltiplas entidades através do parâmetro `entity_name`.
- Os cálculos específicos de cada entidade são executados através das funções:
  - `apply_gold_validation_rules_clientes()`
  - `apply_gold_validation_rules_enderecos()`
- Novos KPIs podem ser adicionados sem necessidade de alterar a estrutura principal do notebook.
- Todos os datasets Gold recebem metadados de auditoria através da função `add_gold_audit_columns()`.
- Os resultados da Gold são posteriormente disponibilizados para consumo em SQL Server, dashboards e ferramentas analíticas.

# CONFIGURAÇÕES GERAIS

## Rodar notebooks de configuração

In [0]:
%run ../../config/feat_squad2_config_adls

In [0]:
%run ../../utils/feat_squad2_utils

## Configurar as variáveis

In [0]:
entity_name = dbutils.widgets.get("entity_name")
folder_name_source = f"silver/ecommerce_{entity_name}"
path_source = f"abfs://{container_name_data_lake}/silver/ecommerce_{entity_name}"
path_gold_base = f"abfs://{container_name_data_lake}/gold"
check_interval = 10 
snapshot_id = 0

storage_options = {
    "storage_account_name": storage_account_name,
    "tenant_id": tenant_id,
    "client_id": client_id,
    "client_secret": client_secret
}

In [0]:
#while True:

snapshot_id += 1

try:

    log.info(f"Iniciando ciclo Gold {snapshot_id}")

    # Ler Silver

    df_silver = read_delta_to_spark_df(
        path=path_source,
        storage_options=storage_options
    )

    log.info(f"Silver carregada.")

    # Aplicar regras de validação da GOLD


    apply_gold_validation_rules(
        entity_name = entity_name,
        df_silver = df_silver
    )
    
    log.info(
        f"Ciclo Gold {snapshot_id} finalizado."
    )

except Exception as e:

    log.error(
        f"Erro no ciclo Gold "
        f"{snapshot_id}: {str(e)}"
    )

#    time.sleep(check_interval)

TESTE

In [0]:
%skip
# Ler gold new customers

df_gold_new_customers = read_delta_to_spark_df(
    path= f"{path_gold_base}/new_customers",
    storage_options=storage_options
)

display(df_gold_new_customers)

In [0]:
%skip
# Ler gold email_provider_rate

df_gold_email_provider_rate = read_delta_to_spark_df(
    path= f"{path_gold_base}/email_provider_rate",
    storage_options=storage_options
)

display(df_gold_email_provider_rate)

In [0]:
%skip
# Ler gold address_distribution_by_state

df_gold_address_distribution_by_state = read_delta_to_spark_df(
    path= f"{path_gold_base}/address_distribution_by_state",
    storage_options=storage_options
)

display(df_gold_address_distribution_by_state)

In [0]:
%skip
# Ler gold first_purchase

df_gold_first_purchase = read_delta_to_spark_df(
    path=f"{path_gold_base}/first_purchase",
    storage_options=storage_options
)

display(df_gold_first_purchase)

In [0]:
%skip
# lista arquivos na gold
all_files = list_files(
                    container_client = container_client_data_lake,
                    folder_name = "gold/address_distribution_by_state"
)                    
print(f"print files in container:")
for files in all_files:
    print(files)

In [0]:
%skip
# Limpar Gold
container_client_data_lake.delete_directory("gold/address_distribution_by_state")

In [0]:
%skip
# Limpar Gold
container_client_data_lake.delete_directory("gold/email_provider_rate")

In [0]:
%skip
# Limpar Gold
container_client_data_lake.delete_directory("gold/new_customers")

In [0]:
%skip
# Limpar Gold
container_client_data_lake.delete_directory("gold/first_purchase")